# PyMessage Walkthrough

A hands-on tour of every function in the package, running entirely against the **built-in example backup** — no real iPhone needed.

Each section is independent. Run them in order for the smoothest experience, or jump straight to whatever interests you.

In [ ]:
from pymessage import (
    EXAMPLE_BACKUP,
    Backup,
    find_backups,
    get_activity_summary,
    get_attachments,
    get_contact_heatmap,
    get_contact_summary,
    get_messages,
    list_conversations,
)

---
## 1. Discovering Backups — `find_backups()`

`find_backups()` scans two places on your Mac:
- `~/Library/Application Support/MobileSync/Backup/` — iPhone backups
- `~/Library/Messages/chat.db` — macOS Messages app

It returns a plain `list[Backup]` sorted by most recent first.

In [ ]:
backups = find_backups()

if backups:
    for b in backups:
        print(b)
else:
    print("No backups found on this machine.")

The `Backup` dataclass wraps everything needed to open a database:

| Field | Description |
|-------|-------------|
| `type` | `"iphone"` or `"macos"` |
| `path` | Backup directory (iphone) or path to `chat.db` (macos) |
| `device_name` | Human-readable label |
| `last_backup` | Datetime of most recent backup |
| `ios_version` | e.g. `"17.2"` |
| `phone_number` | Device phone number |

`EXAMPLE_BACKUP` is a fake iPhone backup built into the package for testing and demos:

In [ ]:
print(EXAMPLE_BACKUP)
print()
print(f"type         = {EXAMPLE_BACKUP.type!r}")
print(f"device_name  = {EXAMPLE_BACKUP.device_name!r}")
print(f"ios_version  = {EXAMPLE_BACKUP.ios_version!r}")
print(f"phone_number = {EXAMPLE_BACKUP.phone_number!r}")
print(f"last_backup  = {EXAMPLE_BACKUP.last_backup}")
print(f"path         = {EXAMPLE_BACKUP.path}")

---
## 2. Getting Messages — `get_messages()`

`get_messages(backup)` queries the iMessage database and returns a DataFrame where every row is one message (or tapback reaction).

In [ ]:
df = get_messages(EXAMPLE_BACKUP)

print(f"Rows:    {len(df)}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# First 8 messages
df[["timestamp", "contact_name", "message_text", "is_from_me", "chat_id"]].head(8)

### `contact_name` — display names from the handle table

- Sent messages (`is_from_me=True`) → `"Me"`
- Received messages → resolved from `handle.uncanonicalized_id` (the display name iMessage stores)

In [ ]:
sent     = df[df["is_from_me"]]
received = df[~df["is_from_me"]]

print(f"Sent messages:     {len(sent):>3}  (contact_name = 'Me')")
print(f"Received messages: {len(received):>3}  (contact_name = sender's display name)")
print()
print("All contacts:", sorted(df["contact_name"].unique().tolist()))

### Tapback reactions

Reactions are stored as separate messages in iMessage. `reaction_type` and `reaction_action` are parsed from `associated_message_type` (codes 2000–2007 = add, 3000–3007 = remove).

In [ ]:
reactions = df[df["reaction_type"].notna()][
    ["contact_name", "reaction_type", "reaction_action", "message_text"]
]
reactions

### `attributedBody` fallback

Modern iOS (16+) stores message text in a binary `attributedBody` column instead of `text`. PyMessage automatically parses it — you never see `None` where there should be text.

In [ ]:
# These messages had NULL in the text column — content came from attributedBody
attr_msgs = df[df["message_text"].str.contains(
    r"just pushed it|Timp trail|Arkansas offer", na=False
)][["contact_name", "message_text"]]

print("Messages decoded from binary attributedBody blobs:")
attr_msgs

---
## 3. Filtering Messages

All filters are optional and composable.

### Filter by phone number

In [ ]:
caleb_df = get_messages(EXAMPLE_BACKUP, phone_numbers="+18015550002")

print(f"Messages with Caleb: {len(caleb_df)}")
caleb_df[["timestamp", "contact_name", "message_text"]].head(6)

### Filter by multiple contacts

In [ ]:
multi_df = get_messages(
    EXAMPLE_BACKUP,
    phone_numbers=["+18015550002", "+18015550003"],  # Caleb + Mitch
)
print(f"Messages with Caleb or Mitch: {len(multi_df)}")
multi_df["contact_name"].value_counts()

### Phone number format flexibility

All of these match the same contact — PyMessage normalises the format before querying.

In [ ]:
for fmt in ["+18015550002", "18015550002", "8015550002"]:
    n = len(get_messages(EXAMPLE_BACKUP, phone_numbers=fmt))
    print(f"  {fmt!r:>15}  →  {n} messages")

### Filter by date range

In [ ]:
# Build a range from the second half of the data
all_ts   = df["timestamp"]
midpoint = all_ts.min() + (all_ts.max() - all_ts.min()) / 2
start    = midpoint.strftime("%Y-%m-%d")
end      = all_ts.max().strftime("%Y-%m-%d")

recent_df = get_messages(EXAMPLE_BACKUP, date_range=(start, end))

print(f"Full data:  {all_ts.min().date()} → {all_ts.max().date()}  ({len(df)} messages)")
print(f"Filtered:   {start} → {end}  ({len(recent_df)} messages)")
recent_df[["timestamp", "contact_name", "message_text"]].head(5)

### Combine filters — phone number + date range

In [ ]:
filtered = get_messages(
    EXAMPLE_BACKUP,
    phone_numbers="+18015550002",
    date_range=(start, end),
)
print(f"Caleb messages in date range: {len(filtered)}")
filtered[["timestamp", "contact_name", "message_text"]]

### Export to CSV

In [ ]:
import os, tempfile

with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
    csv_path = tmp.name

get_messages(EXAMPLE_BACKUP, output_csv=csv_path)
print(f"Exported to: {csv_path}")
print(f"File size:   {os.path.getsize(csv_path) / 1024:.1f} KB")
os.unlink(csv_path)

---
## 4. Listing Conversations — `list_conversations()`

Returns one row per chat with summary stats and a participant list.

In [ ]:
convs = list_conversations(EXAMPLE_BACKUP)

print(f"Conversations: {len(convs)}")
print(f"Columns: {list(convs.columns)}")

In [ ]:
convs[["chat_id", "is_group_chat", "display_name", "participant_count",
       "message_count", "first_message", "last_message"]]

### Group chats — participant lists

In [ ]:
group_chats = convs[convs["is_group_chat"]]

for _, row in group_chats.iterrows():
    print(f"Chat: {row['display_name'] or row['chat_id']}")
    print(f"  Participants ({row['participant_count']}): {row['participants']}")
    print(f"  Messages: {row['message_count']}")

---
## 5. Getting Attachments — `get_attachments()`

Returns metadata for every file shared in messages. For iPhone backups, `file_path` resolves the SHA-1 hashed path inside the backup directory.

In [ ]:
attachments = get_attachments(EXAMPLE_BACKUP)

print(f"Attachments: {len(attachments)}")
print(f"Columns: {list(attachments.columns)}")
attachments

In [ ]:
# Filter attachments to a specific contact
att_filtered = get_attachments(EXAMPLE_BACKUP, phone_numbers="+18015550002")
print(f"Attachments from Caleb: {len(att_filtered)}")

---
## 6. Activity Summary — `get_activity_summary()`

Computes overall statistics across all messages. Returns `(summary_df, top_contacts_df)`.

Optional: `start`, `end`, `last_n_days`, `reference_date`, `top_n`

In [ ]:
summary, top_contacts = get_activity_summary(df)

s = summary.iloc[0]
print("=== Overall Stats ===")
print(f"  Total messages:           {s['total_messages']}")
print(f"  Sent / Received:          {s['total_sent']} / {s['total_received']}")
print(f"  Avg messages/day:         {s['avg_messages_per_day']:.1f}")
print(f"  Unique contacts:          {s['unique_contacts']}")
print(f"  Most active day:          {s['most_active_day_of_week']}")
print(f"  Most active hour (UTC):   {s['most_active_hour']}:00")
print(f"  Avg response time:        {s['avg_response_time_seconds']:.0f}s  ({s['avg_response_time_seconds']/60:.1f} min)")
print(f"  Avg message length:       {s['avg_message_length']:.0f} chars")
print(f"  % messages w/ attachment: {s['pct_messages_with_attachments']:.1%}")
print(f"  Conversations initiated:  {s['conversations_initiated']}")
print(f"  Conversations received:   {s['conversations_received']}")
print(f"  Late-night contacts:      {s['late_night_contacts']}")
print(f"  Ghost contacts:           {s['ghost_contacts']}")

In [ ]:
print("=== Top Contacts ===")
top_contacts

### Filter to last N days

In [ ]:
import pandas as pd

# Use the data's own max timestamp as the reference so the example always works
ref = df["timestamp"].max()

summary_30, top_30 = get_activity_summary(df, last_n_days=30, reference_date=ref)
print(f"Messages in last 30 days: {summary_30.iloc[0]['total_messages']}")
top_30

---
## 7. Per-Contact Stats — `get_contact_summary()`

Returns a single-row DataFrame with detailed stats for one conversation. Accepts the phone number in any format.

In [ ]:
cs = get_contact_summary(df, "+18015550002").iloc[0]  # Caleb

print("=== Caleb ===")
print(f"  Total messages:              {cs['total_messages']}")
print(f"  Sent / Received:             {cs['total_sent']} / {cs['total_received']}")
print(f"  Send-receive ratio:          {cs['send_receive_ratio']:.2f}")
print(f"  Active days:                 {cs['total_active_days']}")
print(f"  Avg messages/active day:     {cs['avg_messages_per_active_day']:.1f}")
print(f"  Avg read time (you):         {cs['avg_read_time_seconds']:.0f}s")
print(f"  Avg response time (you):     {cs['avg_response_time_you_seconds']:.0f}s")
print(f"  Avg response time (contact): {cs['avg_response_time_contact_seconds']:.0f}s")
print(f"  Conversations you started:   {cs['conversations_initiated_you']}")
print(f"  Conversations they started:  {cs['conversations_initiated_contact']}")
print(f"  Longest gap between msgs:    {cs['longest_gap_days']:.1f} days")
print(f"  Messages with attachments:   {cs['messages_with_attachments']}")
print(f"  Avg msg length (you):        {cs['avg_message_length_you']:.0f} chars")
print(f"  Avg msg length (them):       {cs['avg_message_length_contact']:.0f} chars")
print(f"  Short msgs <20 chars (you):  {cs['short_message_count_you']}")
print(f"  Short msgs <20 chars (them): {cs['short_message_count_contact']}")
print(f"  Most active hour:            {cs['most_active_hour']}:00")
print(f"  Most active day:             {cs['most_active_day_of_week']}")

In [ ]:
# Mitch — notable for the 2am message
ms = get_contact_summary(df, "+18015550003").iloc[0]

print("=== Mitch ===")
print(f"  Total messages:   {ms['total_messages']}")
print(f"  Longest gap:      {ms['longest_gap_days']:.1f} days")
print(f"  Most active hour: {ms['most_active_hour']}:00  ← Mitch texted at 2am")

---
## 8. Message Heatmap — `get_contact_heatmap()`

Returns a **7 × 24 DataFrame** of message counts:
- Index: Monday through Sunday  
- Columns: 0–23 (hours in UTC)  
- Values: message counts

In [ ]:
heatmap = get_contact_heatmap(df, "+18015550002")  # Caleb

print(f"Shape: {heatmap.shape}  (7 days × 24 hours)")

# Show only hours with activity
active_hours = heatmap.columns[heatmap.sum() > 0].tolist()
heatmap[active_hours]

### Visualise as a heatmap with seaborn

*(Skip this cell if seaborn isn't installed — `pip install seaborn matplotlib`)*

In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig, ax = plt.subplots(figsize=(14, 4))
    sns.heatmap(
        heatmap,
        ax=ax,
        cmap="YlOrRd",
        linewidths=0.5,
        linecolor="#eee",
        annot=True,
        fmt="d",
        cbar_kws={"label": "Message count"},
    )
    ax.set_title("Caleb — Messages by Day & Hour (UTC)", fontsize=14, pad=12)
    ax.set_xlabel("Hour of day (UTC)")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("seaborn/matplotlib not installed — pip install seaborn matplotlib")
    print("Falling back to plain DataFrame:")
    print(heatmap[active_hours].to_string())

---
## Using Your Own Data

Everything above works identically against a real iPhone backup or macOS Messages database. Just swap `EXAMPLE_BACKUP` for your own:

In [ ]:
# Uncomment and run to use your real data:

# from pymessage import find_backups, get_messages
# backups = find_backups()
# print(backups)          # pick the one you want
#
# df = get_messages(backups[0])
# df.head()

---
## Quick Reference

| Function | Returns | Key optional params |
|----------|---------|---------------------|
| `find_backups()` | `list[Backup]` | — |
| `get_messages(backup)` | `DataFrame` | `phone_numbers`, `date_range`, `output_csv` |
| `list_conversations(backup)` | `DataFrame` | `include_empty` |
| `get_attachments(backup)` | `DataFrame` | `phone_numbers` |
| `get_activity_summary(df)` | `(summary_df, top_df)` | `start`, `end`, `last_n_days`, `top_n` |
| `get_contact_summary(df, contact)` | `DataFrame` | `start`, `end`, `last_n_days` |
| `get_contact_heatmap(df, contact)` | 7×24 `DataFrame` | `start`, `end`, `last_n_days` |